# 1b. Indexing and slicing

About 20 minutes.

By the end: you can pull any piece out of a tensor and say what shape it has
before you run it.
> **Solution.** Read it after your own attempt has failed, then close it and retype the answer from memory.


In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False, linewidth=110)


class _Todo:
    def __repr__(self):
        return "<fill me in>"


TODO = _Todo()


def S(t):
    """Shape as a tuple. Type this constantly."""
    return tuple(t.shape)


def check(label, got, want=None, shape=None, dtype=None, atol=1e-5, rtol=1e-4):
    """Check one answer. Prints what went wrong, then raises."""
    bad = []
    if isinstance(got, _Todo):
        bad.append("not filled in yet")
    else:
        if shape is not None:
            if not torch.is_tensor(got):
                bad.append(f"expected a tensor, got {type(got).__name__}")
            elif S(got) != tuple(shape):
                bad.append(f"shape is {S(got)}, want {tuple(shape)}")
        if dtype is not None and torch.is_tensor(got) and got.dtype != dtype:
            bad.append(f"dtype is {got.dtype}, want {dtype}")
        if want is not None:
            g = got if torch.is_tensor(got) else torch.as_tensor(got)
            w = want if torch.is_tensor(want) else torch.as_tensor(want)
            if S(g) != S(w):
                bad.append(f"shape is {S(g)}, want {S(w)}")
            elif not torch.allclose(g.double(), w.double(), atol=atol, rtol=rtol):
                bad.append(f"values differ, largest gap "
                           f"{(g.double() - w.double()).abs().max().item():.3e}")
    if bad:
        print(f"FAIL  {label}")
        for b in bad:
            print(f"      {b}")
        raise AssertionError(label)
    print(f"ok    {label}")


def same(label, a, b):
    """Two shapes or tuples must match."""
    if isinstance(a, _Todo):
        print(f"FAIL  {label}: not filled in yet")
        raise AssertionError(label)
    if tuple(a) != tuple(b):
        print(f"FAIL  {label}\n      you said {tuple(a)}, it is {tuple(b)}")
        raise AssertionError(label)
    print(f"ok    {label} = {tuple(a)}")

## Two rules

1. An **integer** index removes that axis.
2. A **slice** keeps the axis, with a new length.

In [ ]:
w = torch.arange(32).reshape(4, 8)

print("w[0]    ", S(w[0]), "  int: axis gone")
print("w[0:1]  ", S(w[0:1]), "  slice: axis kept, length 1")
print("w[:, -1]", S(w[:, -1]), "  last column")

## Ellipsis

`...` means "every axis I did not mention". Use it to reach the last axis
without knowing the rank.

In [ ]:
print("(T, 128)        ->", S(torch.zeros(5, 128)[..., :64]))
print("(1, 32, T, 128) ->", S(torch.zeros(1, 32, 5, 128)[..., :64]))

## Step slices de-interleave

`t[0::2]` takes every other row. This is how you split a matrix whose rows
alternate between two things.

In [ ]:
print("even rows, col 0:", w[0::2, 0].tolist())
print("odd rows,  col 0:", w[1::2, 0].tolist())

## Drill 1 (easy)

`qkv` is one token's fused q, k, v output: `H*DH` columns of q, then `HKV*DH`
of k, then `HKV*DH` of v. Slice out the three pieces, keeping the batch axis.

In [ ]:
H, HKV, DH = 4, 2, 2
qkv = torch.arange(H * DH + 2 * HKV * DH).float().reshape(1, -1)

nq, nk = H * DH, HKV * DH
q = qkv[:, :nq]
k = qkv[:, nq:nq + nk]
v = qkv[:, nq + nk:]

check("q", q, shape=(1, H * DH))
check("k", k, shape=(1, HKV * DH))
check("v", v, shape=(1, HKV * DH))
check("v starts after q and k", v[0, 0], want=float(H * DH + HKV * DH))

Note `HKV*DH` is not `H*DH`. BioMistral has 32 query heads and 8 key/value
heads, so k and v are a quarter of q's width. Confusing the two is the single
most common shape bug in this project.

## Drill 2 (medium)

`gu` is a fused gate+up weight matrix stored **interleaved**: row `2i` is gate
row `i`, row `2i+1` is up row `i`. Split it back out.

Then set `is_view` by **testing it in code**, not by typing `True`: does your
`gate` share memory with `gu`?

In [ ]:
I, D = 16, 8
gu = torch.arange(2 * I * D).float().reshape(2 * I, D)

gate = gu[0::2]
up = gu[1::2]
is_view = gate.data_ptr() == gu.data_ptr()

check("gate", gate, shape=(I, D))
check("up", up, shape=(I, D))
check("gate row 0 is gu row 0", gate[0, 0], want=0.0)
check("up row 0 is gu row 1", up[0, 0], want=float(D))
check("is_view", is_view, want=True)

Slices are **views**. They share storage, so writing through one changes the
other. `clone()` if you need a copy.

## Predict

Write each shape as a tuple before you run the cell.

In [ ]:
p = {"a": (8,), "b": (1, 8), "c": (4, 3), "d": (2, 8), "e": (1, 32, 64)}

same("a  w[2]", p["a"], S(w[2]))
same("b  w[2:3]", p["b"], S(w[2:3]))
same("c  w[:, 2:5]", p["c"], S(w[:, 2:5]))
same("d  w[0::2]", p["d"], S(w[0::2]))
same("e  zeros(1,32,128)[..., :64]", p["e"], S(torch.zeros(1, 32, 128)[..., :64]))

## Next

**1c** — changing a tensor's shape without changing its data.